### 初始化，路径+辅助大模型

In [1]:
import warnings
warnings.filterwarnings('ignore')
import json
from pathlib import Path
import time
from synthesis.llm_client import LLMClient
from synthesis.data_utils import save_generated_file, load_json, extract_code_block, verify_python_syntax, verify_json_structure

DOMAIN = "BASE"
llm_model_name = "claude-opus-4-6"
llm = LLMClient(llm_model_name)
# 配置路径
PROMPTS_DIR = Path("prompts") / f"{DOMAIN}"
GENERATED_DATA_DIR = Path("generated") / "data_dir/domains" / f"{DOMAIN}"
GENERATED_CODE_DIR = Path("generated") / "domains" / f"{DOMAIN}"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)

print("✓ 初始化完成")
print(f"LLM: {llm.llm.model_name}")
print(f"Prompts 目录: {PROMPTS_DIR}")
print(f"生成数据目录：{GENERATED_DATA_DIR}")
print(f"生成代码目录：{GENERATED_CODE_DIR}")


def load_schema_and_model():
    """ 如果需要保持schema和model不变 """
    schema_text = None
    model_text = None
    with open(GENERATED_DATA_DIR/ "schema.json", 'r', encoding='utf-8') as f:
        schema_text = f.read()
    with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
        model_text = f.read()
    return schema_text, model_text

✓ 初始化完成
LLM: claude-opus-4-6
Prompts 目录: prompts/BASE
生成数据目录：generated/data_dir/domains/BASE
生成代码目录：generated/domains/BASE


### 0. 生成候选表格以供选择

In [2]:
with open(PROMPTS_DIR / "0_data_table.txt", 'r', encoding='utf-8') as f:
    table = f.read()
response = llm.generate(table)
print(response)
table_text = extract_code_block(response)
print(table_text)

```json
["tenant",
"user",
"user_profile",
"account",
"account_credential",
"auth_provider",
"auth_token",
"auth_session",
"role",
"permission",
"role_permission",
"user_role",
"organization",
"organization_unit",
"organization_unit_closure",
"organization_membership",
"team",
"team_membership",
"project",
"project_membership",
"agent",
"agent_capability",
"agent_permission",
"group",
"group_membership",
"api_key",
"audit_log"]
```
["tenant",
"user",
"user_profile",
"account",
"account_credential",
"auth_provider",
"auth_token",
"auth_session",
"role",
"permission",
"role_permission",
"user_role",
"organization",
"organization_unit",
"organization_unit_closure",
"organization_membership",
"team",
"team_membership",
"project",
"project_membership",
"agent",
"agent_capability",
"agent_permission",
"group",
"group_membership",
"api_key",
"audit_log"]


### 1. 生成schema

In [ ]:
with open(PROMPTS_DIR / "1_data_schema.txt", 'r', encoding='utf-8') as f:
    json_schema_prompt = f.read()
# 已写定，这个无需修改
# tables2cover = " ".join([
#     "user",              # 账号安全：手机号、密码、最后登录时间等
#     "user_profile",      # 个人外显：头像、姓名、简介等
#     "employee",          # 公司身份：工号、入职日期、在职状态、直接主管 ID等
#     "department",        # 组织架构：部门树、部门负责人、部门属性等
#     "department_member", # 员工-部门归属等
#     "position",          # 职级体系：职级（P/M）、岗位序列等
#     "team",              # 虚拟小组：跨部门协作（如办公智能组、HR组）等
#     "team_member",       # 小组成员关系等
#     "role",              # 角色定义（管理员、普通员工、实习生）等
#     "permission",        # 权限原子：具体资源操作权限 (ACL)等
#     "role_permission",   # 角色-权限绑定等
#     "audit_log",         # 操作审计记录等
#     "file_attachment",   # 统一文件引用等
# ])
# json_schema_prompt = json_schema_prompt.replace("{tables2cover}", tables2cover)

print("正在生成 Base JSON Schema")
start_time = time.time()
# use stream mode
schema_response = ""
final_usage = None
for chunk in llm.llm.stream(json_schema_prompt):
    schema_response += chunk.content
    print(chunk.content, end="", flush=True)
    if chunk.usage_metadata:
        final_usage = chunk.usage_metadata
print(f"\n{'-' * 60}")
print(final_usage)
schema_text = extract_code_block(schema_response)
is_json, schema_data = verify_json_structure(schema_text)
if is_json:
    print("✓ JSON 结构验证通过")
    print(f"\nSchema 包含的表：")
    for table_name in schema_data.keys():
        print(f"  - {table_name}")
else:
    print("✗ JSON 结构验证失败，请检查生成的 Schema")

schema_path =  GENERATED_DATA_DIR/ "schema1.json"
save_generated_file(schema_path, schema_text)
print(f"\n✓ JSON Schema 已保存到: {schema_path}")

### 2. 根据schema生成data model

In [ ]:
# 加载 base models prompt 模板
# print(schema_text)
with open(GENERATED_DATA_DIR / "schema1.json", 'r', encoding='utf-8') as f:
    schema_text = f.read()
with open(PROMPTS_DIR / "2_data_models.txt", 'r', encoding='utf-8') as f:
    models_prompt_template = f.read()

# 填充 prompt（将 JSON Schema 传入）
models_prompt = models_prompt_template.replace("{json_schema}", schema_text)

# 调用 LLM 生成 Pydantic models
print("正在基于 JSON Schema 生成 Pydantic Models:")
# use stream mode
models_response = ""
final_usage = None
for chunk in llm.llm.stream(models_prompt):
    models_response += chunk.content
    print(chunk.content, end="", flush=True)
    if chunk.usage_metadata:
        final_usage = chunk.usage_metadata

# print the usage
print(f"\n{'-' * 60}")
print(final_usage)

models_code = extract_code_block(models_response)

# 验证 Python 语法
if verify_python_syntax(models_code):
    print("✓ Python 语法验证通过")
else:
    print("✗ Python 语法验证失败，请检查生成的代码")

# 保存
models_path = GENERATED_CODE_DIR / "models.py"
save_generated_file(models_path, models_code)
print(f"\n✓ Pydantic Models 已保存到: {models_path}")

### 3. 校验生成schema和model的一致性

In [ ]:

validation_prompt = f"""
Schema:
{schema_text}

Data model:
{models_code}

请检查以下三点，如果都满足则回答 "yes"，否则回答 "no" 并说明原因：
1. Data model 中的字段和类型与 Schema 定义一致
2. Data model 中的字段和类型在不同位置保持一致（例如，同一变量在不同地方使用相同的类型）
3. 如果Data model更加合理，可以视为yes

如果是no，给出修改意见
"""

print("正在验证 Schema 和 Model 的一致性:")
# use stream mode
validation_response = ""
final_usage = None
for chunk in llm.llm.stream(validation_prompt):
    validation_response += chunk.content
    print(chunk.content, end="", flush=True)
    if chunk.usage_metadata:
        final_usage = chunk.usage_metadata

print(f"\n验证结果：")
print(validation_response)

In [ ]:
regenerate_prompt = f"""当前的schema为：{schema_text}
当前需要生成的model为：{models_code}
校验结果为：{validation_response}
请根据校验结果修改model
"""
response = llm.generate(regenerate_prompt)
model_code = extract_code_block(response)

# 保存
model_path = GENERATED_CODE_DIR/ "models.py"
save_generated_file(model_path, model_code)

print(f"✓ Data Model 已保存到: {model_path}")  

### 4. 生成数据脚本（Faker，主流程）

让 LLM 生成一个 Python Faker 脚本，由脚本负责实际数据生成：固定 seed 保证可复现、拓扑顺序生成保证 FK 完整性。

In [6]:
import subprocess
import sys
from synthesis.data_utils import parse_ref_fields, topological_sort, validate_foreign_keys, verify_python_syntax

_, models_code = load_schema_and_model()
# 计算拓扑排序和 FK 依赖信息
ref_map = parse_ref_fields(models_code)
all_tables = list(ref_map.keys())
table_order = topological_sort(all_tables, ref_map)
dependency_info = {
    table: [(field, target) for field, target in refs]
    for table, refs in ref_map.items()
}

print("表生成顺序（拓扑排序）：")
for i, t in enumerate(table_order, 1):
    print(f"  {i:2d}. {t}")

# 加载 prompt 并填充变量
with open(PROMPTS_DIR / "3_data_script.txt", 'r', encoding='utf-8') as f:
    script_prompt = f.read()

script_prompt = script_prompt.replace("{model_code}", models_code)
script_prompt = script_prompt.replace("{table_order}", str(table_order))
script_prompt = script_prompt.replace("{dependency_info}", str(dependency_info))

# 调用 LLM 生成数据脚本
print("\n正在生成 BASE 数据脚本...")
print("-" * 60)
full_response = ""
for chunk in llm.llm.stream(script_prompt):
    content = chunk.content
    full_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

# 提取并保存脚本
from synthesis.data_utils import extract_code_block, save_generated_file
script_code = extract_code_block(full_response)
ok, err = verify_python_syntax(script_code)
if not ok:
    print(f"❌ 语法错误：{err}")
else:
    print("✓ Python 语法验证通过")

script_path = GENERATED_CODE_DIR / "gen_data.py"
save_generated_file(str(script_path), script_code)
print(f"✓ 数据脚本已保存到: {script_path}")

表生成顺序（拓扑排序）：
   1. userprofiles
   2. departments
   3. positions
   4. teams
   5. mentorships
   6. roles
   7. permissions
   8. auditlogs
   9. fileattachments
  10. orgconfigs
  11. notifications
  12. employees
  13. rolepermissions
  14. departmentmembers
  15. teammembers
  16. employeeroles

正在生成 BASE 数据脚本...
------------------------------------------------------------


```python
import json
import random
from datetime import datetime, timedelta
from faker import Faker

SEED = 42
random.seed(SEED)
fake = Faker("zh_CN")
Faker.seed(SEED)

result = {}

# Helper functions
def iso_datetime(dt):
    return dt.strftime("%Y-%m-%d %H:%M:%S")

def iso_date(dt):
    return dt.strftime("%Y-%m-%d")

def random_datetime(start, end):
    delta = end - start
    seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=seconds)

BASE_DATE = datetime(2020, 1, 1)
NOW = datetime(2025, 6, 15, 12, 0, 0)

def created_updated(earliest=None, latest=None):
    if ea

In [7]:
# 执行生成脚本，捕获 JSON 输出
print("正在执行数据生成脚本...")
result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True, timeout=120,
    cwd=str(Path(".").resolve())
)

if result.returncode != 0:
    print(f"❌ 脚本执行失败：\n{result.stderr[-2000:]}")
else:
    print("✓ 脚本执行成功")
    from synthesis.data_utils import verify_json_structure, validate_foreign_keys, format_validation_report
    is_json, db_data = verify_json_structure(result.stdout)
    if not is_json:
        print(f"❌ 输出 JSON 解析失败：{db_data}")
    else:
        print(f"✓ JSON 解析成功，共 {len(db_data)} 张表")
        for tname, tdata in db_data.items():
            if isinstance(tdata, dict):
                print(f"  - {tname}: {len(tdata)} 条")

        # FK 完整性验证
        fk_ok, fk_errors = validate_foreign_keys(db_data, ref_map)
        if fk_ok:
            print("✓ 外键完整性验证通过")
        else:
            print(f"⚠️  外键错误 {len(fk_errors)} 处（前5条）：")
            for e in fk_errors[:5]:
                print(f"  - {e}")

        # 保存 db.json
        db_path = GENERATED_DATA_DIR / "db.json"
        save_generated_file(str(db_path), result.stdout)
        print(f"\n✓ BASE Database 已保存到: {db_path}")

正在执行数据生成脚本...
✓ 脚本执行成功
✓ JSON 解析成功，共 16 张表
  - user_profile: 75 条
  - department: 12 条
  - position: 28 条
  - team: 22 条
  - mentorship: 20 条
  - role: 8 条
  - permission: 29 条
  - audit_log: 40 条
  - file_attachment: 25 条
  - org_config: 14 条
  - notification: 40 条
  - employee: 75 条
  - role_permission: 62 条
  - department_member: 75 条
  - team_member: 142 条
  - employee_role: 113 条
✓ 外键完整性验证通过

✓ BASE Database 已保存到: generated/data_dir/domains/BASE/db.json


In [9]:
# 用 Pydantic 模型做最终验证
import json as _json
import sys as _sys
_sys.path.insert(0, str(Path(".").resolve()))
from generated.domains.BASE.models import OfficeDB

with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    db_raw = _json.load(f)

try:
    office_db = OfficeDB.model_validate(db_raw)
    print("✓ OfficeDB 模型验证通过")
    print(office_db.get_statistics())
except Exception as e:
    print(f"❌ 模型验证失败：{e}")

✓ OfficeDB 模型验证通过
{'num_users': 75, 'num_employees': 75, 'num_departments': 12, 'num_teams': 22, 'num_roles': 8, 'num_permissions': 29}


### 备用方案（已丢弃）：直接生成 Database JSON

仅在 Step 4 脚本方式失败时使用。让 LLM 直接输出 JSON，不保证可复现性。

In [ ]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    models_code = f.read()
with open(GENERATED_DATA_DIR/ "schema.json", 'r', encoding='utf-8') as f:
    schema_text = f.read()

# 加载 base database prompt 模板
with open(PROMPTS_DIR / "4_database_content.txt", 'r', encoding='utf-8') as f:
    db_prompt = f.read()

# 填充 prompt
db_prompt = db_prompt.replace("{base_models}", models_code)

# 调用 LLM 生成 database
print("正在生成 Base Database: ")
import time

full_response = ""
chunk_count = 0
start_time = time.time()

print("开始生成数据...")
print("-" * 60)

for chunk in llm.llm.stream(db_prompt):
    content = chunk.content
    full_response += content
    chunk_count += 1
    print(content, end="", flush=True)
    
usage = getattr(chunk, "usage_metadata", None)
print(f"\n{'-' * 60}")
print(usage)
print(f"✅ 生成完成！总计: {chunk_count} chunks, {len(full_response)} 字符")
print(f"⏱️  总耗时: {time.time() - start_time:.2f}秒")
# 提取代码块
db_json_text = extract_code_block(full_response)

# 验证 JSON 结构
is_json, db_data = verify_json_structure(db_json_text)
if db_data:
    print("✓ JSON 结构验证通过")
    
    # 统计数据
    emp_count = len(db_data.get("employees", {}))
    dept_count = len(db_data.get("departments", {}))
    
    print(f"\n数据统计：")
    print(f"  - 员工数量: {emp_count} 人")
    print(f"  - 部门数量: {dept_count} 个")
    
    if emp_count < 10:
        print(f"\n⚠️  警告：员工数量不足 10 人（当前 {emp_count} 人）")
    if dept_count < 5:
        print(f"⚠️  警告：部门数量不足 5 个（当前 {dept_count} 个）")
else:
    print("✗ JSON 结构验证失败，请检查生成的数据")

# 保存
db_path = GENERATED_DATA_DIR/ "db.json"
save_generated_file(db_path, db_json_text)

print(f"\n✓ Base Database 已保存到: {db_path}")


### 5.生成工具

In [12]:
# 加载模型代码
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

# 加载 tools prompt 模板
with open(PROMPTS_DIR / "4_module_tools.txt", 'r', encoding='utf-8') as f:
    tools_prompt = f.read()

tools_prompt = tools_prompt.replace("{model_code}", model_code)

# 调用 LLM 生成工具函数（流式输出）
print("正在生成 BASE 的 Tools...")
tools_response = ""
for chunk in llm.llm.stream(tools_prompt):
    content = chunk.content
    tools_response += content
    print(content, end="", flush=True)

# 提取代码块
tools_code = extract_code_block(tools_response)

# 验证 Python 语法
is_valid, error = verify_python_syntax(tools_code)
if is_valid:
    print("\n✓ Python 语法验证通过")
else:
    print(f"\n✗ Python 语法错误: {error}")

# 保存
tools_path = GENERATED_CODE_DIR / "tools.py"
save_generated_file(tools_path, tools_code)
print(f"✓ Tools 已保存到: {tools_path}")

正在生成 BASE 的 Tools...


```python
"""Toolkit for the office collaboration domain."""
from typing import Dict, List, Optional
import json
from datetime import datetime, timezone

from bench.environment.toolkit import ToolKitBase, ToolType, is_tool
from bench.domains.office.data_model import (
    Department,
    DepartmentMember,
    Employee,
    EmployeeRole,
    Mentorship,
    Notification,
    OrgConfig,
    Permission,
    Position,
    Role,
    RolePermission,
    Team,
    TeamMember,
    UserProfile,
    OfficeDB,
)


class OfficeTools(ToolKitBase):
    """Frequently-used tools for the office collaboration domain."""

    db: OfficeDB

    def __init__(self, db: OfficeDB) -> None:
        super().__init__(db)

    # -----------------------
    # Internal helper methods
    # -----------------------
    def _get_employee(self, employee_ref: str) -> Employee:
        """Get an employee from the database."""
        if employee_ref not in self.db.employee:
            raise ValueE

### 6. 基于已有tools和data content生成Purposes

In [23]:
# 加载已生成的 tools.py
with open(GENERATED_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    tools_code = f.read()
with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_content = f.read()

# 内联 prompt：基于 tools 让 LLM 提议 BASE 域的常见使用目的
purposes_prompt = f"""You are designing benchmark tasks for the BASE domain of an office collaboration system (like DingTalk/Feishu at a Chinese tech company).
Tools of the base domains:
```
{tools_code}
```

Database content:
```
{base_content}
```
Propose up to 20 realistic purposes (stop early if you run out of meaningful ones) from the perspective of actual office workers.
Think from the POV of: HR staff, team lead, IT admin, new employee, or any regular employee.
Hard constraints:
1. Each purpose must be solvable using ONLY the above BASE tools — no HR processes (leave/payroll/onboarding), no IT operations (hardware/software), no admin services (travel/rooms).
2. Each purpose must require at least 2 tool calls to resolve.
3. Prefer action-oriented tasks (more than search) over pure Q&A.
4. Do not generate duplicate (even in semantic meaning) purposes.
5. Make sure the purpose can be solved by current tools and existing data.
Output as JSON list:
[
  {{
    "purpose": "简短目的描述（中文，5字以内）",
    "queries": ["口语化表达1", "口语化表达2"],
    "persona": "谁会有这个需求（例：HR、团队负责人、普通员工）"
  }},
  ...
]
"""

print("正在提议 BASE 域 Purposes...")
# use stream mode
purposes_response = ""
for chunk in llm.llm.stream(purposes_prompt):
    purposes_response += chunk.content
    print(chunk.content, end="", flush=True)
print()
purposes_response = extract_code_block(purposes_response)
is_json, purposes_data = verify_json_structure(purposes_response)
if not is_json:
    print("✗ JSON 结构验证失败，请检查生成的 Schema")
pure_purposes = [x["purpose"] for x in purposes_data]
print(pure_purposes)


正在提议 BASE 域 Purposes...
```json
[
  {
    "purpose": "查看并标已读通知",
    "queries": ["帮我看看我有什么未读通知，然后全部标为已读", "我是EMP042，把我的未读消息都处理一下"],
    "persona": "普通员工"
  },
  {
    "purpose": "查团队成员详情",
    "queries": ["帮我看看核心平台项目组都有谁，把负责人的详细信息告诉我", "数据平台项目组的lead是谁？他是什么岗位的？"],
    "persona": "团队负责人"
  },
  {
    "purpose": "查下属的团队归属",
    "queries": ["帮我查一下王阳参加了哪些团队，分别是什么角色", "我想知道我下属邓莉都在哪些项目组里"],
    "persona": "团队负责人"
  },
  {
    "purpose": "分配角色给员工",
    "queries": ["把团队负责人角色分配给李云，范围是数据平台项目组，我是EMP005来操作", "帮我给朱博分配部门主管角色，scope是后端开发组"],
    "persona": "IT管理员"
  },
  {
    "purpose": "撤销员工角色",
    "queries": ["帮我把席婷婷的实习生角色撤销掉", "EMP070的角色分配需要撤销，帮我处理一下"],
    "persona": "HR"
  },
  {
    "purpose": "查上级汇报链",
    "queries": ["帮我查一下王平的汇报链，一直到最高层是谁", "我想知道李桂芝的直属领导是谁，领导的领导又是谁"],
    "persona": "新员工"
  },
  {
    "purpose": "查部门下属子部门",
    "queries": ["工程技术部下面有哪些子部门？每个子部门的负责人分别是谁？", "帮我看看数据智能部下面有几个组，各自什么状态"],
    "persona": "HR"
  },
  {
    "purpose": "查员工导师关系",
    "queries": ["帮我查一下周红梅的导师是谁，是什么类型的辅导",

### 6b. 生成 Tasks (tasks.json)

In [26]:
import json as _json

with open(PROMPTS_DIR / "5_domain_tasks.txt", 'r', encoding='utf-8') as f:
    task_prompt_template = f.read()

# 人工筛选后填写 purposes 列表（从上一步 LLM 提议中选取）
purposes = ['查看并标已读通知', '查团队成员详情', '查下属的团队归属', '分配角色给员工', '撤销员工角色', '查上级汇报链', '查部门下属子部门', '查员工导师关系', '查岗位级别对比', '统计部门人数', '查角色权限详情', '查公司配置信息', '查员工所属部门', '查某人直接下属', '查特定岗位族员工', '标记单条通知已读', '查实习生分布', '查员工多团队情况', '给新人分配实习角色', '查离职人员信息']

all_tasks = []

for purpose in purposes:
    print(f"\n{'='*60}")
    print(f"正在生成 purpose: {purpose}")
    task_prompt = task_prompt_template
    task_prompt = task_prompt.replace("{tool_code}", tools_code)
    task_prompt = task_prompt.replace("{base_content}", base_content)
    task_prompt = task_prompt.replace("{purpose}", purpose)

    task_response = llm.generate(task_prompt)
    task_text = extract_code_block(task_response)
    print(task_text)

    is_valid, task_data = verify_json_structure(task_text)
    if is_valid:
        # LLM 可能返回单个对象或列表，统一处理为列表
        if isinstance(task_data, dict):
            task_data = [task_data]
        all_tasks.extend(task_data)
        print(f"✓ 生成成功，共 {len(task_data)} 个任务")
    else:
        print(f"✗ JSON 解析失败: {task_data}")
        print(f"  原始输出片段: {task_text[:200]}")
    print('='*60)

# 保存合并后的 tasks.json
tasks_json_text = _json.dumps(all_tasks, ensure_ascii=False, indent=2)
tasks_path = GENERATED_DATA_DIR / "tasks.json"
save_generated_file(tasks_path, tasks_json_text)

print(f"\n✓ 共生成 {len(all_tasks)} 个任务，已保存到: {tasks_path}")


正在生成 purpose: 查看并标已读通知
{
    "id": "notif_read_001",
    "description": {
      "purpose": "查看并标已读通知",
      "relevant_policies": null,
      "notes": "用户想查看自己的未读通知，了解内容后将所有未读通知标记为已读"
    },
    "user_scenario": {
      "persona": "房玉珍，基础架构组的一名基础架构工程师",
      "instructions": {
        "task_instructions": "你是房玉珍，你想查看自己有哪些未读通知，了解通知内容后，把所有未读通知都标记为已读。",
        "reason_for_call": "最近工作比较忙，好几天没看系统通知了，想集中处理一下未读通知，看看有没有重要的审批请求或公告。",
        "known_info": "你的名字是房玉珍，你在基础架构组工作。",
        "unknown_info": "你不知道自己的employee_ref，也不知道具体有多少条未读通知。",
        "domain": "BASE"
      }
    },
    "initial_state": null,
    "evaluation_criteria": {
      "actions": [
        {
          "arguments": {
            "display_name": "房玉珍"
          },
          "name": "find_employee_ref_by_name",
          "action_id": "action_1"
        },
        {
          "arguments": {
            "employee_ref": "EMP024",
            "unread_only": true
          },
          "name": "list_notifications_for_employee",
